In [ ]:
# CRISPRedict: Prediction and Explainability on Knockout Dataset

This notebook performs prediction and SHAP-based explainability using a pre-trained CRISPRedict model on the Knockout dataset. The analysis includes:
- Automatic detection of sequence column
- Feature extraction based on CRISPRedict logic
- Score prediction using a trained statsmodels regressor
- SHAP-based feature importance explanation
- Saving results with prediction scores and SHAP values


In [ ]:
#setup and imports
import os
import pandas as pd
import pickle
import shap
from statsmodels.tools import add_constant

# Import the feature extraction function (from local file 'features.py')
from features import seq_features


In [ ]:
#Load Dataset and Model (Markdown + Code)
## Load Input Data and Pre-trained Model
# Set paths (adjust if needed)
data_path = r"C:\Users\faiza\30nt crispredict scores data.xlsx"
model_path = r"C:\Users\faiza\Downloads\Downloaded meterials crisperedict repository\CRISPRedict_u6_regressor.pickle"

# Load Knockout dataset
df = pd.read_excel(data_path)
print("Loaded dataset with shape:", df.shape)



In [ ]:
## Feature Extraction from Knockout Sequences
# Detect sequence column
def detect_sequence_column(df):
    for col in df.columns:
        if "sequence" in col.lower():
            return col
    raise KeyError("No sequence column found!")

# Feature extraction function
def extract_features(data, sequence_col, promoter='u6'):
    results = []
    for _, row in data.iterrows():
        sequence = row[sequence_col]
        features = seq_features(sequence, single=True, promoter=promoter)
        features.insert(0, 'Sequence', sequence)
        results.append(features)
    return pd.concat(results, ignore_index=True)

# Run feature extraction
sequence_col = detect_sequence_column(df)
features_df = extract_features(df, sequence_col, promoter='u6')



In [ ]:
## Predict Scores Using CRISPRedict Model
# Load pre-trained statsmodels model
with open(model_path, 'rb') as model_file:
    model = pickle.load(model_file)

# Expected features (from CRISPRedict paper)
expected_features = [
    'GC binary', 'AG count', 'AA count', 'TTT count', 'AAA count',
    'ATT count', 'CTT count', 'G_24', 'G_21', 'G_20', 'C_20', 'C_19',
    'C_18', 'C_3', 'C_1', 'A_16', 'A_13', 'A_middle', 'G_middle', 'T_end',
    'TG_end', 'AC_end', 'CA_19', 'CT_13', 'GT_pos', 'CCC_18', 'Motifs',
    'Polynucleotides'
]

# Prepare data
features = features_df[expected_features]
features = add_constant(features, has_constant='add')

# Predict scores
features_df['Prediction scores'] = model.predict(features)


In [ ]:
## SHAP-Based Feature Importance Analysis
# SHAP wrapper
class StatsmodelsModelWrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        return self.model.predict(X)

wrapped_model = StatsmodelsModelWrapper(model)

# Run SHAP explainability
explainer = shap.Explainer(wrapped_model.predict, features)
shap_values = explainer(features)

# Waterfall plot for first sequence
shap.plots.waterfall(shap_values[0], max_display=20)


In [ ]:
## Save Predictions and SHAP Values to Excel
# Add SHAP values to DataFrame
all_features = ['const'] + expected_features
for i, feature_name in enumerate(all_features):
    features_df[f"SHAP_{feature_name}"] = shap_values.values[:, i]

# Round SHAP values
for col in features_df.columns:
    if 'SHAP_' in col:
        features_df[col] = features_df[col].round(5)

# Save to Excel
output_path = os.path.join(os.path.expanduser("~"), "Downloads", "crispredict_knockout_results_with_shap.xlsx")
features_df.to_excel(output_path, index=False)

print(f" Results saved to: {output_path}")


In [ ]:
# Show first 5 rows
features_df[['Sequence', 'Prediction scores'] + [col for col in features_df.columns if 'SHAP_' in col]].head()


In [ ]:
## Note on Environment

This notebook was developed in Python 3.7.12 using the CRISPRedict model environment.  
For full dependency versions, see `requirements.txt` or the `README.md`.


In [ ]:
#Full finall code structured

import os
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tools import add_constant
from features import seq_features

# Function to detect the sequence column dynamically
def detect_sequence_column(df):
    for col in df.columns:
        if "sequence" in col.lower():  
            return col
    raise KeyError("No sequence column found! Ensure a column contains 'sequence' in its name.")

# Feature extraction function (using features.py logic)
def extract_features(data, sequence_col, promoter='u6'):
    results = []
    for _, row in data.iterrows():
        sequence = row[sequence_col]
        features = seq_features(sequence, single=True, promoter=promoter)
        features.insert(0, 'Sequence', sequence)
        results.append(features)
    return pd.concat(results, ignore_index=True)

# Main process function
def process_and_predict(data_path, model_path, promoter='u6'):
    print("Loading data...")
    df = pd.read_excel(data_path)

    # Detect the correct column name
    sequence_col = detect_sequence_column(df)
    print(f"Detected sequence column: {sequence_col}")

    print("Extracting features...")
    features_df = extract_features(df, sequence_col, promoter=promoter)

    print("Loading trained model...")
    with open(model_path, 'rb') as model_file:
        model = pickle.load(model_file)

    print("Preprocessing features...")
    # Select the features expected by the model( GC binary (Authers method),Dinucleotide and trinucleotide counts (Authers method)
    expected_features = [
        'GC binary', 'AG count', 'AA count', 'TTT count', 'AAA count',
        'ATT count', 'CTT count', 'G_24', 'G_21', 'G_20', 'C_20', 'C_19', # Nucleotide position-specific features (Authers method)
        'C_18', 'C_3', 'C_1', 'A_16', 'A_13', 'A_middle', 'G_middle', 'T_end', # # Positional features (Authers method)
        'TG_end', 'AC_end', 'CA_19', 'CT_13', 'GT_pos', 'CCC_18', 'Motifs', # Motif-based features  (Authers method)
        'Polynucleotides'                                                    # Count polynucleotides (Authers method)
    ]
    features = features_df[expected_features]

    # Add a constant column manually
    features = add_constant(features, has_constant='add')

    # Debugging: Print feature shape
    print("Feature matrix shape:", features.shape)

    print("Predicting scores...")
    try:
        predictions = model.predict(features)
        features_df['Prediction scores'] = predictions
    except ValueError as e:
        print("Error during prediction:", e)
        print("Check if features align with the model's expected input.")
        raise

    # Save results
    output_path = os.path.join(os.path.expanduser("~"), "Downloads", "prediction_scores_with_expected_features.xlsx")
    features_df.to_excel(output_path, index=False)
    print(f"Results saved at: {output_path}")

    return features_df

# File paths
data_path = r'C:\Users\faiza\Downloads\All excel files training testing\Data sets for all tools with Results\30nt crispredict scores data.xlsx'
model_path = r'C:\Users\faiza\Downloads\Downloaded meterials crisperedict repository\CRISPRedict_u6_regressor.pickle'

# Run the process with the chosen promoter type ('u6' or 't7')
results_df = process_and_predict(data_path, model_path, promoter='u6')

# Print results preview
print("Preview of results with prediction scores and features:")
print(results_df.head())
